# 第11周 · 练习：手算注意力、掩码与 shape

四个小练习：在纸上算一次注意力；实现带掩码的 softmax；给多头注意力的每一步标出 shape；亲手验证"没有位置编码，打乱顺序也看不出来"。

In [ ]:
# 运行这一格来安装课程工具包（每次在 Colab 打开 notebook 后运行一次）
import importlib.util, subprocess, sys
if importlib.util.find_spec("zero2researcher") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ReaganGen/Zero2Researcher"], check=True)
print("课程工具包已就绪")

In [ ]:
import math

import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from zero2researcher.checks import week11 as ck
from zero2researcher.checks import report

## E1 · 手算注意力

3 个 token、$d_k = 4$（$\sqrt{d_k} = 2$）：

$$Q = \begin{bmatrix} 0&1&0&1 \\ 2&0&2&0 \\ 1&0&0&1 \end{bmatrix},\quad K = \begin{bmatrix} 1&0&1&0 \\ 1&1&0&0 \\ 0&0&1&1 \end{bmatrix},\quad V = \begin{bmatrix} 2&0 \\ 0&2 \\ 1&1 \end{bmatrix}$$

**先在纸上**（可以用计算器）算 **token 2** 的三个注意力权重 $w_{2j}$ 和输出向量，照讲义里的表格：点积 $q_2 \cdot k_j$ → 除以 2 → 求 $e$ 的幂 → 归一化 → 对 $V$ 的三行加权平均。把结果（保留 3 位小数）填进下面的变量。

In [ ]:
# 在这里写你的代码
raise NotImplementedError
ck.hand_attention(weights_2, out_2)

算完再用 NumPy 核对一遍（这一格不计分）。然后回答：token 2 为什么最关注 token 1？

In [ ]:
Q = np.array([[0, 1, 0, 1], [2, 0, 2, 0], [1, 0, 0, 1]], dtype=float)
K = np.array([[1, 0, 1, 0], [1, 1, 0, 0], [0, 0, 1, 1]], dtype=float)
V = np.array([[2, 0], [0, 2], [1, 1]], dtype=float)
# 在这里写你的代码
raise NotImplementedError


（在这里回答）

## E2 · 带掩码的 softmax

实现 `masked_softmax(scores, mask)`：`mask` 与 `scores` 同形状，1（或 True）= 保留，0（或 False）= 遮住。要求被遮住的位置概率**恰好为 0**，且每行剩下的概率之和仍为 1。

想一想：为什么不能把被遮住的分数填成 0？为什么不能先 softmax 再乘以 mask？（下一格会演示这两种错法。）

<details>
<summary>卡住了再看：提示 1</summary>

被遮住的位置要在 softmax **之前**改成某个数，让 $e$ 的这个数次方恰好是 0——是哪个数？

</details>

<details>
<summary>还是卡住：提示 2</summary>

`scores.masked_fill(mask == 0, float("-inf"))`，然后沿最后一维（`dim=-1`）softmax。

</details>

In [ ]:
def masked_softmax(scores, mask):
    """scores: (..., T)；mask: 同形状，1 = 保留、0 = 遮住"""
    # 在这里写你的代码
    raise NotImplementedError


ck.masked_softmax(masked_softmax)

In [ ]:
# 对照两种常见的错误写法，看看它们错在哪里（这一格不用改）
scores = torch.tensor([[2.0, 1.0, 0.5, -1.0]])
mask = torch.tensor([[1, 1, 0, 0]])
wrong_fill_zero = F.softmax(scores.masked_fill(mask == 0, 0.0), dim=-1)
wrong_multiply = F.softmax(scores, dim=-1) * mask
print("填 0 再 softmax:     ", wrong_fill_zero.numpy().round(3), " ← 被遮住的位置仍分到概率")
print("先 softmax 再乘 mask:", wrong_multiply.numpy().round(3), " ← 行和 =", round(wrong_multiply.sum().item(), 3))

## E3 · shape 标注：多头注意力的每一步

下面是多头注意力前向的 7 行核心代码（B = 2、T = 7、C = 12、H = 3，所以每个头 d = 4）。**先别运行**，只读代码：

1. 在每行代码末尾的注释里写出这一行结果的 shape（课程规矩，以后每次都要写）；
2. 把同样的 shape 用具体数字写成元组，填进 `predicted`。

然后运行下一格，它会真的执行这 7 行并和你的预测逐个比对。

In [ ]:
B, T, C, H = 2, 7, 12, 3
d = C // H
torch.manual_seed(0)
x = torch.randn(B, T, C)
qkv_proj = nn.Linear(C, 3 * C)

# qkv = qkv_proj(x)                                    # shape: ?
# qkv_heads = qkv.reshape(B, T, H, 3 * d)              # shape: ?
# qkv_perm = qkv_heads.permute(0, 2, 1, 3)             # shape: ?
# q, k, v = qkv_perm.chunk(3, dim=-1)                  # q 的 shape: ?
# scores = q @ k.transpose(-2, -1) / math.sqrt(d)      # shape: ?
# values = torch.softmax(scores, dim=-1) @ v           # shape: ?
# merged = values.permute(0, 2, 1, 3).reshape(B, T, C) # shape: ?

predicted = {}
# 在这里写你的代码
raise NotImplementedError


In [ ]:
qkv = qkv_proj(x)
qkv_heads = qkv.reshape(B, T, H, 3 * d)
qkv_perm = qkv_heads.permute(0, 2, 1, 3)
q, k, v = qkv_perm.chunk(3, dim=-1)
scores = q @ k.transpose(-2, -1) / math.sqrt(d)
values = torch.softmax(scores, dim=-1) @ v
merged = values.permute(0, 2, 1, 3).reshape(B, T, C)

actual = {"qkv": qkv, "qkv_heads": qkv_heads, "qkv_perm": qkv_perm, "q": q,
          "scores": scores, "values": values, "merged": merged}
wrong = [name for name, t in actual.items() if tuple(predicted.get(name, ())) != tuple(t.shape)]
for name, t in actual.items():
    print(f"{name:10s} 预测 {str(tuple(predicted.get(name, ()))):16s} 实际 {tuple(t.shape)}")
report("多头注意力的 shape 标注", not wrong, f"这几步写错了：{wrong}",
       "permute(0, 2, 1, 3) 交换第 1、2 维；chunk(3, dim=-1) 把最后一维切成 3 等份；"
       "q @ kᵀ 的结果是 (…, T, T)")

## E4 · 打乱顺序：验证置换等变

下面给出讲义里写过的 `EncoderBlock` 与 `PositionalEncoding`（照 UvA Tutorial 6）。任务：

1. 构造一个 `EncoderBlock(input_dim=8, num_heads=2, dim_feedforward=16)` 并 `.eval()`；输入 `x` 的 shape 为 `(1, 10, 8)`，随机顺序 `perm = torch.randperm(10)`；
2. 算 `diff_without_pe` = $\max |f(x_{\text{perm}}) - f(x)_{\text{perm}}|$（不加位置编码）；
3. 算 `diff_with_pe`：两边都先加位置编码再过块，$\max |f(PE(x_{\text{perm}})) - f(PE(x))_{\text{perm}}|$。

In [ ]:
def scaled_dot_product(q, k, v, mask=None):
    d_k = q.size(-1)
    attn_logits = q @ k.transpose(-2, -1) / math.sqrt(d_k)
    if mask is not None:
        attn_logits = attn_logits.masked_fill(mask == 0, float("-inf"))
    attention = F.softmax(attn_logits, dim=-1)
    return attention @ v, attention


class MultiheadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.embed_dim, self.num_heads = embed_dim, num_heads
        self.head_dim = embed_dim // num_heads
        self.qkv_proj = nn.Linear(embed_dim, 3 * embed_dim)
        self.o_proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x, mask=None, return_attention=False):
        B, T, C = x.shape
        qkv = self.qkv_proj(x).reshape(B, T, self.num_heads, 3 * self.head_dim).permute(0, 2, 1, 3)
        q, k, v = qkv.chunk(3, dim=-1)
        values, attention = scaled_dot_product(q, k, v, mask=mask)
        o = self.o_proj(values.permute(0, 2, 1, 3).reshape(B, T, C))
        return (o, attention) if return_attention else o


class EncoderBlock(nn.Module):
    def __init__(self, input_dim, num_heads, dim_feedforward, dropout=0.0):
        super().__init__()
        self.self_attn = MultiheadAttention(input_dim, num_heads)
        self.linear_net = nn.Sequential(nn.Linear(input_dim, dim_feedforward), nn.Dropout(dropout),
                                        nn.ReLU(inplace=True), nn.Linear(dim_feedforward, input_dim))
        self.norm1, self.norm2 = nn.LayerNorm(input_dim), nn.LayerNorm(input_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        x = self.norm1(x + self.dropout(self.self_attn(x, mask=mask)))
        return self.norm2(x + self.dropout(self.linear_net(x)))


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0), persistent=False)

    def forward(self, x):
        return x + self.pe[:, : x.size(1)]

In [ ]:
torch.manual_seed(1)
# 在这里写你的代码
raise NotImplementedError
print(f"无位置编码 {diff_without_pe:.2e}   有位置编码 {diff_with_pe:.3f}")
ck.permutation_equivariance(diff_without_pe, diff_with_pe)

**问题**（两三句话）：举一个"顺序无关紧要、置换等变反而正合适"的任务，再举一个"没有位置编码就一定做不好"的任务。

（在这里回答）